In [ ]:
# Demonstration of a Quantum Support Vector Machine example using PennyLane AI
# This particular model uses 4 qubits in an attempt to improve classificaiton accuracy
# Developed by: Dr. Michael P. Haydock - IBM Fellow Emeritus
# Visiting Professor of Mathematics, Computer Science & Statistics - St. Olaf College
# Initial Coding: 4/24/2025

# Import necessary libraries
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:


# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic dataset
X = np.random.rand(200, 2) * 2 - 1  # Generate random points between -1 and 1
Y = np.where(X[:, 0] * X[:, 1] > 0, 1, 0)  # Classify based on quadrant logic

# Split dataset into training and testing sets (80% training, 20% testing)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

# Standardize features for better performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define quantum feature map with increased qubits (4 instead of 2)
n_qubits = 4  # Increased qubit count for more expressive power
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_kernel(x1, x2):
    """
    Quantum kernel function with 4 qubits.
    Uses a feature embedding and entanglement for better class separation.
    """
#    qml.Hadamard(wires=range(n_qubits))  # Apply Hadamard gates for superposition

    for i in range(n_qubits):
        qml.Hadamard(wires=i)  # Correct (applying Hadamard to each wire separately)
    
    # Encode classical data into quantum states
    qml.AngleEmbedding(x1, wires=range(n_qubits))
    
    # Add entanglement using CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])  # Entangle adjacent qubits
    
    qml.AngleEmbedding(x2, wires=range(n_qubits))  # Encode second input

    # Add more entanglement layers for better feature mapping
    for i in range(n_qubits - 1, 0, -1):
        qml.CNOT(wires=[i, i - 1])  # Reverse entanglement direction
    
    return qml.expval(qml.PauliZ(0))  # Expectation value measures similarity

# Compute quantum kernel matrices
quantum_kernel_matrix = np.array([[quantum_kernel(x1, x2) for x2 in X_train] for x1 in X_train])

# Train classical SVM using the quantum kernel
svm = SVC(kernel="precomputed", gamma=0.5)  # Adjusted gamma parameter for better separability
svm.fit(quantum_kernel_matrix, Y_train)

# Compute kernel matrix for test data and predict labels
quantum_kernel_matrix_test = np.array([[quantum_kernel(x1, x2) for x2 in X_train] for x1 in X_test])
predictions = svm.predict(quantum_kernel_matrix_test)

# Evaluate model performance
accuracy = sum(predictions == Y_test) / len(Y_test)
print(f"QSVM Accuracy with 4 Qubits: {accuracy:.2f}")

# Visualize quadrant classification
plt.scatter(X_train[Y_train == 0][:, 0], X_train[Y_train == 0][:, 1], label="Class 0", color="blue")
plt.scatter(X_train[Y_train == 1][:, 0], X_train[Y_train == 1][:, 1], label="Class 1", color="red")
plt.axhline(0, color="black", linestyle="--")  # Add horizontal axis
plt.axvline(0, color="black", linestyle="--")  # Add vertical axis
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("QSVM Classification with 4 Qubits")
plt.show()

# Print quantum circuit for verification
print(qml.draw(quantum_kernel)([0.5, -0.3], [0.1, 0.7]))